In [127]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("GPU not detected")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [128]:
import requests
import pandas as pd

url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

params = {
    "$limit": 100000,
    "$where": "created_date >= '2026-03-01T00:00:00'",
    "$order": "created_date ASC"
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

df = pd.DataFrame(data)

print("Dataset shape:", df.shape)

print("\nDate range:")
print("Oldest :", df["created_date"].min())
print("Latest :", df["created_date"].max())

print("\nStatus distribution:")
print(df["status"].value_counts())

df.head()

Dataset shape: (100000, 44)

Date range:
Oldest : 2026-03-01T00:00:07.000
Latest : 2026-03-09T23:15:38.000

Status distribution:
status
Closed         96628
Open            1347
In Progress     1280
Pending          466
Assigned         211
Started           67
Unspecified        1
Name: count, dtype: int64


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,location_type,incident_zip,incident_address,...,descriptor_2,bridge_highway_name,bridge_highway_segment,vehicle_type,facility_type,bridge_highway_direction,road_ramp,taxi_pick_up_location,taxi_company_borough,due_date
0,68166342,2026-03-01T00:00:07.000,2026-03-01T02:49:39.000,NYPD,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,Street/Sidewalk,11230,855 EAST 19 STREET,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,68163601,2026-03-01T00:00:20.000,2026-03-01T00:35:08.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Talking,Street/Sidewalk,10468,36 EAST BEDFORD PARK BOULEVARD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,68163320,2026-03-01T00:00:36.000,2026-03-01T00:58:36.000,NYPD,New York City Police Department,Blocked Driveway,No Access,Street/Sidewalk,10457,2161 CLINTON AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,68166371,2026-03-01T00:00:40.000,2026-03-01T02:43:43.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,11216,1161 DEAN STREET,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,68157778,2026-03-01T00:00:52.000,2026-03-01T00:56:08.000,NYPD,New York City Police Department,Noise - Residential,Loud Talking,Residential Building/House,10009,540 EAST 5 STREET,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [129]:
print("Shape:", df.shape)

print("\nColumn names and data types:")
print(df.dtypes.to_string())

Shape: (100000, 44)

Column names and data types:
unique_key                        object
created_date                      object
closed_date                       object
agency                            object
agency_name                       object
complaint_type                    object
descriptor                        object
location_type                     object
incident_zip                      object
incident_address                  object
street_name                       object
cross_street_1                    object
cross_street_2                    object
intersection_street_1             object
intersection_street_2             object
address_type                      object
city                              object
landmark                          object
status                            object
resolution_description            object
resolution_action_updated_date    object
community_board                   object
council_district                  object
police_

In [88]:
missing_analysis = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percentage': (df.isnull().mean() * 100).round(2),
    'unique_values': df.nunique(dropna=True)
})

missing_analysis = missing_analysis.sort_values(
    'missing_percentage',
    ascending=False
)

missing_analysis

TypeError: unhashable type: 'dict'

In [89]:
for col in df.columns:
    dict_count = df[col].apply(lambda x: isinstance(x, dict)).sum()
    list_count = df[col].apply(lambda x: isinstance(x, list)).sum()

    if dict_count > 0 or list_count > 0:
        print(
            f"{col}: "
            f"{dict_count} dictionaries, "
            f"{list_count} lists"
        )

location: 95005 dictionaries, 0 lists


In [90]:
missing_analysis = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percentage': (df.isnull().mean() * 100).round(2),
})

# Calculate unique values safely
unique_values = {}

for col in df.columns:
    if col == 'location':
        # Location is a dictionary, so count unique representations
        unique_values[col] = df[col].dropna().apply(str).nunique()
    else:
        unique_values[col] = df[col].nunique(dropna=True)

missing_analysis['unique_values'] = pd.Series(unique_values)

missing_analysis = missing_analysis.sort_values(
    'missing_percentage',
    ascending=False
)

missing_analysis

,missing_count,missing_percentage,unique_values
taxi_company_borough,99905,99.90,5
due_date,99669,99.67,241
taxi_pick_up_location,99146,99.15,619
road_ramp,99055,99.06,31
bridge_highway_direction,99040,99.04,71
bridge_highway_name,98876,98.88,57
bridge_highway_segment,98875,98.88,367
vehicle_type,95210,95.21,8
facility_type,86847,86.85,2
descriptor_2,51907,51.91,542


In [91]:
core_columns = [
    'unique_key',
    'created_date',
    'closed_date',
    'agency',
    'complaint_type',
    'descriptor',
    'borough',
    'status',
    'resolution_description'
]

df_core = df[core_columns].copy()

print("Core dataset shape:", df_core.shape)
df_core.head()

Core dataset shape: (100000, 9)


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,status,resolution_description
0,68166342,2026-03-01T00:00:07.000,2026-03-01T02:49:39.000,NYPD,Illegal Parking,Posted Parking Sign Violation,BROOKLYN,Closed,The New York City Police Department responded ...
1,68163601,2026-03-01T00:00:20.000,2026-03-01T00:35:08.000,NYPD,Noise - Street/Sidewalk,Loud Talking,BRONX,Closed,The New York City Police Department responded ...
2,68163320,2026-03-01T00:00:36.000,2026-03-01T00:58:36.000,NYPD,Blocked Driveway,No Access,BRONX,Closed,The New York City Police Department responded ...
3,68166371,2026-03-01T00:00:40.000,2026-03-01T02:43:43.000,NYPD,Noise - Residential,Loud Music/Party,BROOKLYN,Closed,The New York City Police Department responded ...
4,68157778,2026-03-01T00:00:52.000,2026-03-01T00:56:08.000,NYPD,Noise - Residential,Loud Talking,MANHATTAN,Closed,The New York City Police Department responded ...


In [92]:
print("Shape:", df_core.shape)

print("\nData types:")
print(df_core.dtypes)

print("\nMissing values:")
print(df_core.isnull().sum())

Shape: (100000, 9)

Data types:
unique_key                object
created_date              object
closed_date               object
agency                    object
complaint_type            object
descriptor                object
borough                   object
status                    object
resolution_description    object
dtype: object

Missing values:
unique_key                   0
created_date                 0
closed_date               3169
agency                       0
complaint_type               0
descriptor                   0
borough                      0
status                       0
resolution_description     870
dtype: int64


In [93]:
status_counts = df_core['status'].value_counts()

status_percentage = (
    df_core['status']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

status_summary = pd.DataFrame({
    'count': status_counts,
    'percentage': status_percentage
})

status_summary

,count,percentage
status,,
Closed,96623,96.62
Open,1349,1.35
In Progress,1281,1.28
Pending,466,0.47
Assigned,212,0.21
Started,68,0.07
Unspecified,1,0.00


In [94]:
status_resolution_check = pd.crosstab(
    df_core['status'],
    df_core['resolution_description'].isna(),
    margins=True
)

status_resolution_check.columns = [
    'Resolution Available',
    'Resolution Missing',
    'Total'
]

status_resolution_check

,Resolution Available,Resolution Missing,Total
status,,,
Assigned,212,0,212
Closed,96440,183,96623
In Progress,616,665,1281
Open,1328,21,1349
Pending,466,0,466
Started,68,0,68
Unspecified,0,1,1
All,99130,870,100000


In [95]:
complaint_counts = (
    df_core['complaint_type']
    .value_counts()
    .head(20)
)

complaint_counts

,count
complaint_type,
Illegal Parking,15332
Noise - Residential,10236
HEAT/HOT WATER,9794
Street Condition,7296
Blocked Driveway,4938
Traffic Signal Condition,3300
Noise - Street/Sidewalk,3211
UNSANITARY CONDITION,3004
Dirty Condition,2647


In [96]:
print("Unique complaint types:", df_core['complaint_type'].nunique())
print("Unique descriptors:", df_core['descriptor'].nunique())

print("\nTop 30 descriptors:")
print(df_core['descriptor'].value_counts().head(30))

Unique complaint types: 161
Unique descriptors: 677

Top 30 descriptors:
descriptor
Loud Music/Party                  9134
ENTIRE BUILDING                   6457
Pothole                           5661
Blocked Hydrant                   4328
Banging/Pounding                  3922
Posted Parking Sign Violation     3918
No Access                         3754
APARTMENT ONLY                    3337
Trash                             2321
Controller                        2261
With License Plate                2099
Blocked Sidewalk                  1973
Loud Talking                      1756
PESTS                             1464
N/A                               1320
Partial Access                    1184
Derelict Vehicles                 1114
Blocked Crosswalk                 1030
Commercial Overnight Parking       942
MOLD                               941
Double Parked Blocking Traffic     924
CEILING                            845
Removal Request                    836
Cave-in            

In [97]:
for col in ['complaint_type', 'descriptor', 'borough', 'resolution_description']:
    print(f"\n--- {col} ---")
    print("Missing:", df_core[col].isna().sum())
    print("N/A:", (df_core[col].astype(str).str.strip().str.upper() == 'N/A').sum())
    print("Unique:", df_core[col].nunique())


--- complaint_type ---
Missing: 0
N/A: 0
Unique: 161

--- descriptor ---
Missing: 0
N/A: 1320
Unique: 677

--- borough ---
Missing: 0
N/A: 0
Unique: 6

--- resolution_description ---
Missing: 870
N/A: 453
Unique: 470


In [98]:
resolution_counts = (
    df_core['resolution_description']
    .dropna()
    .value_counts()
)

print("Unique non-null resolutions:", resolution_counts.shape[0])

print("\nTop 30 resolution descriptions:")
for i, (resolution, count) in enumerate(resolution_counts.head(30).items(), 1):
    print(f"{i:2}. [{count:5}] {resolution}")

Unique non-null resolutions: 470

Top 30 resolution descriptions:
 1. [10723] The New York City Police Department responded to the complaint and with the information available observed no evidence of a criminal violation at that time. If the problem persists, please contact 311 to create another complaint. If possible, provide contact information so responding officers may reach out to you for more details. If necessary, your complaint may be referred to your local precinct's special operations units (Quality of Life, etc.). We count on New Yorkers like yourself to maintain a safe City, so please let us know if you see other conditions that require our attention.
 2. [ 9040] The New York City Police Department responded to the complaint and their investigation determined that no criminal violation existed. The condition was corrected without the need to issue a summons or effect an arrest. If the problem persists, please contact 311 to create another complaint. If possible, provide con

In [99]:
resolution_diversity = (
    df_core[df_core['resolution_description'].notna()]
    .groupby('complaint_type')['resolution_description']
    .nunique()
    .sort_values(ascending=False)
)

print("Complaint types with most distinct resolutions:")
print(resolution_diversity.head(50))

Complaint types with most distinct resolutions:
complaint_type
Street Condition                           41
Sidewalk Condition                         40
Sewer                                      31
Consumer Complaint                         29
General Construction/Plumbing              26
Curb Condition                             23
Building/Use                               22
Obstruction                                21
WATER LEAK                                 19
Water System                               17
PAINT/PLASTER                              17
UNSANITARY CONDITION                       17
Damaged Tree                               15
HEAT/HOT WATER                             15
Special Projects Inspection Team (SPIT)    15
Noise                                      15
Derelict Vehicles                          14
Maintenance or Facility                    14
Street Sign - Damaged                      14
Dirty Condition                            14
Indoor Air Qualit

In [100]:
# 1. Number of unique complaint types
num_complaint_types = df_core['complaint_type'].nunique()

# 2. Number of unique descriptors for each complaint type
descriptors_per_complaint = (
    df_core.groupby('complaint_type')['descriptor']
    .nunique()
    .sort_values(ascending=False)
)

# 3. Number of unique resolutions for each descriptor
resolutions_per_descriptor = (
    df_core[df_core['resolution_description'].notna()]
    .groupby('descriptor')['resolution_description']
    .nunique()
    .sort_values(ascending=False)
)

print("Number of unique complaint types:", num_complaint_types)

print("\nTop 30 complaint types by number of descriptors:")
print(descriptors_per_complaint.head(30))

print("\nTop 30 descriptors by number of distinct resolutions:")
print(resolutions_per_descriptor.head(30))

Number of unique complaint types: 161

Top 30 complaint types by number of descriptors:
complaint_type
Consumer Complaint                59
Traffic Signal Condition          41
General Construction/Plumbing     27
Street Light Condition            27
Food Establishment                25
Street Condition                  18
Sidewalk Condition                16
Sewer                             14
Illegal Parking                   14
Noise                             13
Street Sign - Damaged             13
Water System                      13
Street Sign - Dangling            13
Maintenance or Facility           11
Street Sign - Missing             11
Highway Condition                 11
Broken Parking Meter               9
Water Quality                      9
Air Quality                        9
Residential Disposal Complaint     9
PLUMBING                           8
Water Conservation                 8
Dead Animal                        8
Animal in a Park                   7
Obstructi

In [101]:
combo_resolution = (
    df_core[df_core['resolution_description'].notna()]
    .groupby(['complaint_type', 'descriptor'])['resolution_description']
    .nunique()
    .sort_values(ascending=False)
)

print("Total complaint + descriptor combinations:", len(combo_resolution))

print("\nTop 30 combinations by resolution diversity:")
print(combo_resolution.head(30))

Total complaint + descriptor combinations: 748

Top 30 combinations by resolution diversity:
complaint_type                           descriptor                                           
Sidewalk Condition                       Broken Sidewalk                                          26
Street Condition                         Cave-in                                                  22
Building/Use                             Illegal Conversion Of Residential Building/Space         18
Street Condition                         Rough, Pitted or Cracked Roads                           18
PAINT/PLASTER                            CEILING                                                  17
Curb Condition                           Broken Curb                                              17
Street Condition                         Failed Street Repair                                     16
UNSANITARY CONDITION                     MOLD                                                     16
PAIN

In [102]:
combo_stats = (
    df_core[df_core['resolution_description'].notna()]
    .groupby(['complaint_type', 'descriptor'])
    .agg(
        total_cases=('unique_key', 'count'),
        unique_resolutions=('resolution_description', 'nunique')
    )
    .sort_values(
        ['unique_resolutions', 'total_cases'],
        ascending=[False, False]
    )
)

print("Total combinations:", len(combo_stats))

print("\nTop 30 combinations:")
print(combo_stats.head(30))

Total combinations: 748

Top 30 combinations:
                                                                                            total_cases  \
complaint_type                          descriptor                                                        
Sidewalk Condition                      Broken Sidewalk                                             312   
Street Condition                        Cave-in                                                     716   
Building/Use                            Illegal Conversion Of Residential Building/Space            296   
Street Condition                        Rough, Pitted or Cracked Roads                              162   
PAINT/PLASTER                           CEILING                                                     845   
Curb Condition                          Broken Curb                                                 128   
UNSANITARY CONDITION                    MOLD                                                      

In [103]:
resolution_distribution = pd.cut(
    combo_stats['unique_resolutions'],
    bins=[0, 1, 3, 5, 10, float('inf')],
    labels=[
        '1 resolution',
        '2-3 resolutions',
        '4-5 resolutions',
        '6-10 resolutions',
        '11+ resolutions'
    ]
)

distribution_counts = resolution_distribution.value_counts().sort_index()

distribution_percent = (
    distribution_counts / len(combo_stats) * 100
).round(2)

resolution_summary = pd.DataFrame({
    'combinations': distribution_counts,
    'percentage': distribution_percent
})

resolution_summary

,combinations,percentage
unique_resolutions,,
1 resolution,241,32.22
2-3 resolutions,175,23.40
4-5 resolutions,117,15.64
6-10 resolutions,172,22.99
11+ resolutions,43,5.75


In [104]:
# For each complaint + descriptor combination,
# calculate how many cases belong to each resolution.
combo_resolution_counts = (
    df_core[df_core['resolution_description'].notna()]
    .groupby(
        ['complaint_type', 'descriptor', 'resolution_description']
    )
    .size()
    .reset_index(name='resolution_cases')
)

print("Total complaint-descriptor-resolution combinations:",
      len(combo_resolution_counts))

print("\nTop 30 most frequent resolution outcomes:")
print(
    combo_resolution_counts
    .sort_values('resolution_cases', ascending=False)
    .head(30)
)

Total complaint-descriptor-resolution combinations: 3088

Top 30 most frequent resolution outcomes:
                complaint_type                     descriptor  \
2506          Street Condition                        Pothole   
1685       Noise - Residential               Loud Music/Party   
1164            HEAT/HOT WATER                ENTIRE BUILDING   
2730  Traffic Signal Condition                     Controller   
2500          Street Condition                        Pothole   
1271           Illegal Parking                Blocked Hydrant   
1683       Noise - Residential               Loud Music/Party   
1675       Noise - Residential               Banging/Pounding   
1270           Illegal Parking                Blocked Hydrant   
151           Blocked Driveway                      No Access   
1154            HEAT/HOT WATER                ENTIRE BUILDING   
150           Blocked Driveway                      No Access   
1712   Noise - Street/Sidewalk               Loud Music

In [105]:
# Count cases for every complaint + descriptor + resolution
resolution_support = (
    df_core[df_core['resolution_description'].notna()]
    .groupby(
        ['complaint_type', 'descriptor', 'resolution_description']
    )
    .size()
    .reset_index(name='resolution_cases')
)

# Number of resolutions with at least 5 historical examples
supported_resolution_counts = (
    resolution_support
    .assign(supported=lambda x: x['resolution_cases'] >= 5)
    .groupby(['complaint_type', 'descriptor'])
    .agg(
        total_resolutions=('resolution_description', 'nunique'),
        resolutions_with_5plus_cases=('supported', 'sum')
    )
)

supported_resolution_counts['support_ratio'] = (
    supported_resolution_counts['resolutions_with_5plus_cases']
    / supported_resolution_counts['total_resolutions']
)

print("Top 30 complaint-descriptor combinations by supported resolution diversity:")
print(
    supported_resolution_counts
    .sort_values(
        ['resolutions_with_5plus_cases', 'total_resolutions'],
        ascending=False
    )
    .head(30)
)

Top 30 complaint-descriptor combinations by supported resolution diversity:
                                                                                total_resolutions  \
complaint_type                descriptor                                                            
HEAT/HOT WATER                ENTIRE BUILDING                                                  15   
Street Condition              Cave-in                                                          22   
Dirty Condition               Trash                                                            14   
Street Condition              Rough, Pitted or Cracked Roads                                   18   
PAINT/PLASTER                 WALL                                                             15   
Derelict Vehicles             Derelict Vehicles                                                14   
HEAT/HOT WATER                APARTMENT ONLY                                                   12   
Street Conditio

In [106]:
usable_cases = df_core[
    (df_core['status'] == 'Closed') &
    (df_core['resolution_description'].notna()) &
    (df_core['resolution_description'].str.strip().str.upper() != 'N/A')
].copy()

print("Total records:", len(df_core))
print("Usable historical cases:", len(usable_cases))
print("Percentage usable:", round(len(usable_cases) / len(df_core) * 100, 2), "%")

Total records: 100000
Usable historical cases: 95987
Percentage usable: 95.99 %


In [107]:
rag_columns = [
    'complaint_type',
    'descriptor',
    'borough',
    'resolution_description'
]

rag_cases = usable_cases[rag_columns].copy()

print("RAG cases:", len(rag_cases))

print("\nExact duplicate rows:")
print(rag_cases.duplicated().sum())

print("\nUnique case representations:")
print(rag_cases.drop_duplicates().shape[0])

RAG cases: 95987

Exact duplicate rows:
88749

Unique case representations:
7238


In [108]:
pattern_stats = {
    "Unique complaint + descriptor + resolution": (
        usable_cases[
            ['complaint_type', 'descriptor', 'resolution_description']
        ].drop_duplicates().shape[0]
    ),

    "Unique complaint + descriptor + borough + resolution": (
        usable_cases[
            ['complaint_type', 'descriptor', 'borough', 'resolution_description']
        ].drop_duplicates().shape[0]
    )
}

for key, value in pattern_stats.items():
    print(f"{key}: {value}")

Unique complaint + descriptor + resolution: 2929
Unique complaint + descriptor + borough + resolution: 7238


In [109]:
resolution_stats = (
    usable_cases['resolution_description']
    .value_counts()
    .reset_index()
)

resolution_stats.columns = ['resolution_description', 'case_count']

print("Unique resolution descriptions:", len(resolution_stats))

print("\nResolution frequency distribution:")
print(resolution_stats.head(50).to_string(index=False))

Unique resolution descriptions: 439

Resolution frequency distribution:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      resolution_description  case_count
                                                                                                           The New York City Police Department responded to the complaint and with the information available observed n

In [110]:
resolution_counts = (
    usable_cases['resolution_description']
    .value_counts()
)

for n in [5, 10, 20, 50, 100]:
    covered = resolution_counts.head(n).sum()
    percentage = covered / len(usable_cases) * 100

    print(
        f"Top {n:3} resolutions: "
        f"{covered:5} cases "
        f"({percentage:.2f}%)"
    )

Top   5 resolutions: 38059 cases (39.65%)
Top  10 resolutions: 54378 cases (56.65%)
Top  20 resolutions: 69814 cases (72.73%)
Top  50 resolutions: 84722 cases (88.26%)
Top 100 resolutions: 91565 cases (95.39%)


In [111]:
df_core['created_date'] = pd.to_datetime(df_core['created_date'])

monthly_counts = (
    df_core
    .groupby(df_core['created_date'].dt.to_period('M'))
    .size()
)

print("Monthly ticket counts:")
print(monthly_counts)

print("\nNumber of months:", len(monthly_counts))

Monthly ticket counts:
created_date
2026-03    100000
Freq: M, dtype: int64

Number of months: 1


In [112]:
#new findings

In [113]:
# Measure the variation added by descriptor_2

usable = df.dropna(subset=['resolution_description']).copy()

# Without descriptor_2
base_cols = [
    'complaint_type',
    'descriptor',
    'resolution_description'
]

# With descriptor_2
d2_cols = [
    'complaint_type',
    'descriptor',
    'descriptor_2',
    'resolution_description'
]

base_unique = usable.drop_duplicates(subset=base_cols).shape[0]
d2_unique = usable.drop_duplicates(subset=d2_cols).shape[0]

print("Total usable records:", len(usable))
print()

print("WITHOUT descriptor_2:")
print("Unique representations:", base_unique)
print("Duplicate records:", len(usable) - base_unique)

print()

print("WITH descriptor_2:")
print("Unique representations:", d2_unique)
print("Duplicate records:", len(usable) - d2_unique)

print()

print("Additional unique representations from descriptor_2:",
      d2_unique - base_unique)

print("Percentage increase:",
      round((d2_unique - base_unique) / base_unique * 100, 2), "%")

Total usable records: 99130

WITHOUT descriptor_2:
Unique representations: 3088
Duplicate records: 96042

WITH descriptor_2:
Unique representations: 4330
Duplicate records: 94800

Additional unique representations from descriptor_2: 1242
Percentage increase: 40.22 %


In [114]:
# How much does descriptor_2 differentiate the same complaint + descriptor?

d2_variation = (
    usable.groupby(['complaint_type', 'descriptor'])['descriptor_2']
    .nunique(dropna=True)
    .reset_index(name='unique_descriptor_2')
)

print("Complaint + descriptor combinations:", len(d2_variation))

print("\nDistribution of descriptor_2 variation:")
print(d2_variation['unique_descriptor_2'].value_counts().sort_index())

print("\nCombinations with multiple descriptor_2 values:")
print(
    d2_variation[d2_variation['unique_descriptor_2'] > 1]
    .sort_values('unique_descriptor_2', ascending=False)
    .head(20)
)

Complaint + descriptor combinations: 748

Distribution of descriptor_2 variation:
unique_descriptor_2
0     149
1     402
2      91
3      33
4      26
5      12
6       8
7       7
8       4
9       6
10      6
11      1
13      2
16      1
Name: count, dtype: int64

Combinations with multiple descriptor_2 values:
                 complaint_type                          descriptor  \
370     Maintenance or Facility                Structure - Outdoors   
113          Consumer Complaint              Other Store (Non-Food)   
77           Consumer Complaint  Bodega, Deli, or Convenience Store   
103          Consumer Complaint                      Hotel or Motel   
146                Damaged Tree      Branch or Limb Has Fallen Down   
316             Illegal Parking         Parking Permit Improper Use   
363     Maintenance or Facility               Graffiti or Vandalism   
633              Taxi Complaint        Driver Complaint - Passenger   
663    Traffic Signal Condition             

In [115]:
# Does descriptor_2 create different resolution outcomes?

resolution_variation = (
    usable.groupby(
        ['complaint_type', 'descriptor', 'descriptor_2']
    )['resolution_description']
    .nunique()
    .reset_index(name='unique_resolutions')
)

print("Total complaint + descriptor + descriptor_2 groups:",
      len(resolution_variation))

print("\nDistribution of number of resolutions:")
print(
    resolution_variation['unique_resolutions']
    .value_counts()
    .sort_index()
)

print("\nGroups with multiple resolutions:")
print(
    resolution_variation[
        resolution_variation['unique_resolutions'] > 1
    ]
    .sort_values('unique_resolutions', ascending=False)
    .head(20)
)

Total complaint + descriptor + descriptor_2 groups: 1143

Distribution of number of resolutions:
unique_resolutions
1     536
2     173
3      99
4      88
5      53
6      38
7      35
8      50
9      20
10     18
11      8
12      5
13      5
14      6
15      3
16      4
18      1
22      1
Name: count, dtype: int64

Groups with multiple resolutions:
            complaint_type                                        descriptor  \
829       Street Condition                                           Cave-in   
841       Street Condition                    Rough, Pitted or Cracked Roads   
800     Sidewalk Condition                                   Broken Sidewalk   
801     Sidewalk Condition                                   Broken Sidewalk   
1068  UNSANITARY CONDITION                                              MOLD   
833       Street Condition                              Failed Street Repair   
1136          Water System                         Leak (Use Comments) (WA2)   
799

In [116]:
# Compare resolution diversity before and after adding descriptor_2

# WITHOUT descriptor_2
without_d2 = (
    usable.groupby(['complaint_type', 'descriptor'])
    ['resolution_description']
    .nunique()
)

# WITH descriptor_2
with_d2 = (
    usable.groupby(['complaint_type', 'descriptor', 'descriptor_2'])
    ['resolution_description']
    .nunique()
)

print("WITHOUT descriptor_2")
print("Groups:", len(without_d2))
print("Groups with multiple resolutions:",
      (without_d2 > 1).sum())
print("Percentage with multiple resolutions:",
      round((without_d2 > 1).mean() * 100, 2), "%")

print("\nWITH descriptor_2")
print("Groups:", len(with_d2))
print("Groups with multiple resolutions:",
      (with_d2 > 1).sum())
print("Percentage with multiple resolutions:",
      round((with_d2 > 1).mean() * 100, 2), "%")

WITHOUT descriptor_2
Groups: 748
Groups with multiple resolutions: 507
Percentage with multiple resolutions: 67.78 %

WITH descriptor_2
Groups: 1143
Groups with multiple resolutions: 607
Percentage with multiple resolutions: 53.11 %


In [117]:
print("Total usable records:", len(usable))

print("Records with descriptor_2:",
      usable['descriptor_2'].notna().sum())

print("Records missing descriptor_2:",
      usable['descriptor_2'].isna().sum())

print("Coverage:",
      round(usable['descriptor_2'].notna().mean() * 100, 2), "%")

Total usable records: 99130
Records with descriptor_2: 47368
Records missing descriptor_2: 51762
Coverage: 47.78 %


In [118]:
# Descriptor_2 coverage by complaint type

d2_coverage = (
    usable.groupby('complaint_type')
    .agg(
        total_cases=('descriptor_2', 'size'),
        with_descriptor_2=('descriptor_2', 'count')
    )
)

d2_coverage['coverage_percent'] = (
    d2_coverage['with_descriptor_2']
    / d2_coverage['total_cases']
    * 100
)

# Add volume so we can see both coverage and importance
d2_coverage = d2_coverage.sort_values(
    'total_cases',
    ascending=False
)

print(d2_coverage.head(30))

                               total_cases  with_descriptor_2  \
complaint_type                                                  
Illegal Parking                      15332                616   
Noise - Residential                  10236                  0   
HEAT/HOT WATER                        9794               9794   
Street Condition                      7296               1651   
Blocked Driveway                      4938                  0   
Traffic Signal Condition              3300               3300   
Noise - Street/Sidewalk               3211                  0   
UNSANITARY CONDITION                  3004               3004   
Dirty Condition                       2605               2605   
Abandoned Vehicle                     2099                  0   
PLUMBING                              2039               2039   
PAINT/PLASTER                         1743               1743   
Noise                                 1582               1582   
Noise - Commercial       

In [119]:
# How many actual records remain duplicated after using descriptor_2?

group_cols = [
    'complaint_type',
    'descriptor',
    'descriptor_2',
    'resolution_description'
]

group_sizes = (
    usable.groupby(group_cols, dropna=False)
    .size()
)

print("Total usable records:", len(usable))
print("Unique representations:", len(group_sizes))
print()

print("Representations occurring once:",
      (group_sizes == 1).sum())

print("Representations occurring more than once:",
      (group_sizes > 1).sum())

print()

print("Records belonging to duplicated representations:",
      group_sizes[group_sizes > 1].sum())

print("Percentage of records in duplicated representations:",
      round(
          group_sizes[group_sizes > 1].sum()
          / len(usable) * 100,
          2
      ), "%")

Total usable records: 99130
Unique representations: 4330

Representations occurring once: 1556
Representations occurring more than once: 2774

Records belonging to duplicated representations: 97574
Percentage of records in duplicated representations: 98.43 %


In [120]:
# Measure diversity of the actual input/query representation
# WITHOUT using the resolution as an input feature

input_cols = [
    'complaint_type',
    'descriptor',
    'descriptor_2'
]

input_groups = (
    usable.groupby(input_cols, dropna=False)
    .size()
    .sort_values(ascending=False)
)

print("Total usable records:", len(usable))
print("Unique input representations:", len(input_groups))
print()

print("Input representations occurring once:",
      (input_groups == 1).sum())

print("Input representations occurring more than once:",
      (input_groups > 1).sum())

print()

print("Top 20 most frequent input representations:")
print(input_groups.head(20))

Total usable records: 99130
Unique input representations: 1306

Input representations occurring once: 332
Input representations occurring more than once: 974

Top 20 most frequent input representations:
complaint_type            descriptor                     descriptor_2            
Street Condition          Pothole                        NaN                         5633
Noise - Residential       Loud Music/Party               NaN                         5560
Illegal Parking           Blocked Hydrant                NaN                         4328
                          Posted Parking Sign Violation  NaN                         3918
Blocked Driveway          No Access                      NaN                         3754
Noise - Residential       Banging/Pounding               NaN                         3739
HEAT/HOT WATER            ENTIRE BUILDING                NO HEAT                     3140
Noise - Street/Sidewalk   Loud Music/Party               NaN                         

In [121]:
input_cols = ['complaint_type', 'descriptor', 'descriptor_2']
pattern_stats = (
    usable.groupby(input_cols)
    .agg(
        total_cases=('resolution_description', 'size'),
        unique_resolutions=('resolution_description', 'nunique'),
    )
)
# Also compute top-1 share
top1_share = (
    usable.groupby(input_cols)['resolution_description']
    .apply(lambda s: s.value_counts(normalize=True).iloc[0])
)
pattern_stats['top1_share'] = top1_share

In [122]:
pattern_stats.head(20)

total_cases  \
complaint_type   descriptor                                  descriptor_2                                  
APPLIANCE        ELECTRIC/GAS RANGE                          CONTROL KNOB BROKEN OR MISSING           45   
                                                             NO GAS                                    3   
                                                             NO GAS OR ELECTRICITY                   196   
                                                             NO STOVE KNOB COVERS                     10   
                                                             OVEN DOOR MISSING OR BROKEN              59   
                                                             PILOT LIGHT OUT                          54   
                                                             RANGE/OVEN BROKEN OR MISSING              3   
                 REFRIGERATOR                                BROKEN DOOR SEAL                        207   
                                                             BROKEN OR MISSING                         3   
Abandoned Bike   Chained to Public Property                  N/A                                      55   
Air Quality      Air: Dust, Construction/Demolition (AE4)    AE4                                      19   
                 Air: Dust, Other (Use Comments) (AE5)       AE5                                       2   
                 Air: Odor, Nail Salon (AD8)                 AD8                                       2   
                 Air: Odor, Sweet From Unknown Source (AZ1)  AZ1                                       1   
                 Air: Odor/Fumes, Restaurant (AD2)           AD2                                      17   
                 Air: Odor/Fumes, Vehicle Idling (AD3)       AD3                                     152   
                 Air: Other Air Problem (Use Comments) (AZZ) AZZ                                       5   
                 Air: Smoke, Chimney or vent (AS1)           AS1                                      14   
                 Air: Smoke, Vehicular (AA4)                 AA4                                       2   
Animal in a Park Animal Waste                                Dog                                      16   

                                                                                             unique_resolutions  \
complaint_type   descriptor                                  descriptor_2                                         
APPLIANCE        ELECTRIC/GAS RANGE                          CONTROL KNOB BROKEN OR MISSING                   5   
                                                             NO GAS                                           3   
                                                             NO GAS OR ELECTRICITY                            8   
                                                             NO STOVE KNOB COVERS                             5   
                                                             OVEN DOOR MISSING OR BROKEN                      6   
                                                             PILOT LIGHT OUT                                  7   
                                                             RANGE/OVEN BROKEN OR MISSING                     2   
                 REFRIGERATOR                                BROKEN DOOR SEAL                                 9   
                                                             BROKEN OR MISSING                                3   
Abandoned Bike   Chained to Public Property                  N/A                                             11   
Air Quality      Air: Dust, Construction/Demolition (AE4)    AE4                                              2   
                 Air: Dust, Other (Use Comments) (AE5)       AE5                                              1   
                 Air: Odor, Nail Salon (AD8)                 AD8                                          

In [123]:
print("Total input patterns:", len(pattern_stats))

print("\nTop-1 share distribution:")
print(
    pd.cut(
        pattern_stats["top1_share"],
        bins=[0, 0.25, 0.50, 0.75, 0.90, 1.0],
        include_lowest=True
    ).value_counts().sort_index()
)

print("\nResolution diversity:")
print(
    pattern_stats["unique_resolutions"]
    .value_counts()
    .sort_index()
)

print("\nPatterns with multiple resolutions:",
      (pattern_stats["unique_resolutions"] > 1).sum())

print("Patterns with 3+ resolutions:",
      (pattern_stats["unique_resolutions"] >= 3).sum())


Total input patterns: 1143

Top-1 share distribution:
top1_share
(-0.001, 0.25]     31
(0.25, 0.5]       338
(0.5, 0.75]       152
(0.75, 0.9]        60
(0.9, 1.0]        562
Name: count, dtype: int64

Resolution diversity:
unique_resolutions
1     536
2     173
3      99
4      88
5      53
6      38
7      35
8      50
9      20
10     18
11      8
12      5
13      5
14      6
15      3
16      4
18      1
22      1
Name: count, dtype: int64

Patterns with multiple resolutions: 607
Patterns with 3+ resolutions: 434


In [124]:
pattern_stats['volume'] = pattern_stats['total_cases']
pattern_stats['bucket'] = pd.cut(
    pattern_stats['top1_share'],
    bins=[0, 0.25, 0.5, 0.75, 0.9, 1.0],
    labels=['very_diverse', 'diverse', 'moderate', 'near_det', 'deterministic']
)

# Volume-weighted distribution
volume_by_bucket = pattern_stats.groupby('bucket')['volume'].sum()
print(volume_by_bucket / volume_by_bucket.sum() * 100)

bucket
very_diverse     12.905337
diverse          56.998396
moderate         12.016551
near_det          5.060378
deterministic    13.019338
Name: volume, dtype: float64


/tmp/ipykernel_2114/3937412227.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  volume_by_bucket = pattern_stats.groupby('bucket')['volume'].sum()


In [125]:
pattern_stats.groupby('bucket')['total_cases'].describe()

/tmp/ipykernel_2114/771868197.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pattern_stats.groupby('bucket')['total_cases'].describe()


,count,mean,std,min,25%,50%,75%,max
bucket,,,,,,,,
very_diverse,31.0,197.193548,359.253895,4.0,11.0,98.0,204.00,1884.0
diverse,338.0,79.878698,256.613173,2.0,4.0,15.0,53.75,3140.0
moderate,152.0,37.447368,76.176406,3.0,4.0,10.0,36.25,580.0
near_det,60.0,39.950000,96.080726,5.0,8.0,15.5,33.25,718.0
deterministic,562.0,10.973310,81.280468,1.0,1.0,1.0,3.00,1701.0


In [126]:
pattern_stats.nlargest(20, 'total_cases')[['complaint_type','descriptor','descriptor_2','total_cases','top1_share']]

KeyError: "['complaint_type', 'descriptor', 'descriptor_2'] not in index"

In [ ]:
closed_df = df[df["status"] == "Closed"].copy()

print("Closed tickets:", len(closed_df))
print("Shape:", closed_df.shape)

closed_df.to_csv("nyc_311_closed_tickets.csv", index=False)

print("Saved as: nyc_311_closed_tickets.csv")

In [130]:
columns = [
    "unique_key",
    "created_date",
    "closed_date",
    "complaint_type",
    "descriptor",
    "descriptor_2",
    "resolution_description"
]

closed_tickets = df[df["status"] == "Closed"][columns].copy()

closed_tickets.to_csv("closed_tickets_rag.csv", index=False)

print("Saved:", len(closed_tickets), "closed tickets")
print("Columns:", list(closed_tickets.columns))

Saved: 96628 closed tickets
Columns: ['unique_key', 'created_date', 'closed_date', 'complaint_type', 'descriptor', 'descriptor_2', 'resolution_description']
